**for large data on hugging face**

In [ ]:
from datasets import load_dataset
import os
import re

# 1. Load the dataset
print("Downloading large movie dataset...")
dataset = load_dataset("AIatMongoDB/embedded_movies", split="train")

# 2. Setup your movie_data folder
output_dir = "./movie_data_large"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 3. Save as individual files (NTCIR Style)
print("Processing and saving files...")
for i, item in enumerate(dataset):
    # Skip records with missing essential data
    if not item.get('fullplot') or not item.get('title'):
        continue

    title = item['title']
    plot = item['fullplot']

    # FIX: Use 'or []' to handle None values gracefully
    directors = item.get('directors') or []
    cast = item.get('cast') or []

    # Sanitize title for filename
    safe_title = re.sub(r'[^\w\s]', '', title).strip().replace(' ', '_')
    filename = f"{output_dir}/{safe_title}_{i}.txt"

    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"MOVIE TITLE: {title}\n")
            f.write(f"DIRECTORS: {', '.join(directors)}\n")
            f.write(f"CAST: {', '.join(cast)}\n")
            f.write(f"STORY PLOT: {plot}")
    except Exception as e:
        print(f"Skipping {title} due to error: {e}")

print(f"Success! Your library now contains {len(os.listdir(output_dir))} movie files.")

Implimenting using Langchain and gimini-2.5 flash and pinecone vector database

Now using this because Gemini cause a single point of failure when pushing data in pinecone, so we ar currently using-all-mpnet-base-v2 transformer model

In [ ]:
pip install -U langgraph langchain-pinecone langchain-google-genai pinecone-client

In [ ]:
!pip install -U langchain-community langchain-text-splitters

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install -U langchain-huggingface sentence-transformers

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
import os
import time
import re
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. SETUP PINECONE KEY
os.environ["PINECONE_API_KEY"] = "pinecone_api_key_here"  # <-- REPLACE with your actual Pinecone API key
P_KEY = os.environ["PINECONE_API_KEY"]
index_name = "movie-index"

# --- HELPER: EXTRACT YEAR, GENRE, TITLE ---
def extract_rich_metadata(text, filename):
    meta = {
        "title": os.path.basename(filename).replace('.txt', ''),
        "year": 0,
        "genre": "Unknown"
    }

    # 1. Extract Year (Looks for 19xx or 20xx)
    year_match = re.search(r'(19|20)\d{2}', text[:1000]) # Search first 1000 chars
    if year_match:
        meta["year"] = int(year_match.group())

    # 2. Basic Genre Tagging
    genres = ["Noir", "Comedy", "Sci-Fi", "Horror", "Crime", "Action", "Drama", "Thriller"]
    for g in genres:
        if g.lower() in text[:1000].lower():
            meta["genre"] = g
            break

    return meta

# 2. LOAD & PROCESS DATA
print("📂 Loading movies from storage...")
if not os.path.exists('./movie_data_large'):
    print("❌ Error: Directory './movie_data_large' not found!")
else:
    loader = DirectoryLoader('./movie_data_large', glob="./*.txt", loader_cls=TextLoader)
    raw_documents = loader.load()
    print(f"📄 Loaded {len(raw_documents)} files.")

    # --- APPLY METADATA TO RAW DOCUMENTS ---
    print("🧠 Extracting rich metadata (Year, Genre, Title)...")
    for doc in raw_documents:
        source_file = doc.metadata.get('source', '')
        enriched_meta = extract_rich_metadata(doc.page_content, source_file)
        doc.metadata.update(enriched_meta)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    docs = text_splitter.split_documents(raw_documents)
    print(f"✂️ Created {len(docs)} chunks with rich metadata.")

    # 3. INITIALIZE LOCAL EMBEDDINGS
    print("🤖 Initializing Local Transformer (all-mpnet-base-v2)...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # 4. UPLOAD TO PINECONE
    print(f"🚀 Starting Upload to: {index_name}...")
    BATCH_SIZE = 100

    try:
        vectorstore = PineconeVectorStore.from_documents(
            docs[0:BATCH_SIZE],
            embeddings,
            index_name=index_name,
            pinecone_api_key=P_KEY
        )

        
        for i in range(BATCH_SIZE, len(docs), BATCH_SIZE):
            batch = docs[i : i + BATCH_SIZE]
            print(f"Syncing Batch {i//BATCH_SIZE + 1} of {len(docs)//BATCH_SIZE + 1}...")
            vectorstore.add_documents(batch)
            time.sleep(1) 

        print("✅ SUCCESS: Your index is now metadata-aware!")

    except Exception as e:
        print(f"❌ System Failure: {e}")

In [ ]:
# Test Search: Only show 1970s Crime movies
results = vectorstore.similarity_search(
    "A protagonist tries to do right but it goes wrong",
    k=3,
    filter={
        "year": {"$gte": 1970, "$lt": 1980},
        "genre": "Crime"
    }
)

for d in results:
    print(f"Found: {d.metadata['title']} ({d.metadata['year']})")

In [ ]:
!pip install groq

Implimenting Self Correction Loop: Lang Graph

In [ ]:
!pip install langgraph

In [ ]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, END
from groq import Groq

# Initialize Groq client (set your API key in env)
client = Groq(api_key="grok_api_key_here")

# 1. DEFINE STATE
class GraphState(TypedDict):
    query: str
    context: str
    prediction: str
    relevance_score: str
    iterations: int


# 2. NODES

# 🔍 Retrieve Node
def retrieve_node(state: GraphState):
    print("\n--- RETRIEVING FROM VECTOR DB ---")

    docs = vectorstore.similarity_search(state["query"], k=3)
    context = "\n".join([d.page_content for d in docs])

    return {
        "context": context,
        "iterations": state.get("iterations", 0) + 1
    }


# 🧠 Grade Node (LLM decides relevance)
def grade_node(state: GraphState):
    print("\n--- GRADING RELEVANCE ---")

    prompt = f"""
    You are a grader.

    Query: {state['query']}
    Context: {state['context']}

    Is this context relevant to answer the query?
    Answer only: YES or NO
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )

    score = response.choices[0].message.content.strip().lower()

    return {"relevance_score": score}


# ✏️ Rewrite Query Node
def rewrite_node(state: GraphState):
    print("\n--- REWRITING QUERY ---")

    prompt = f"""
    The following query did not retrieve relevant results.

    Original Query: {state['query']}

    Rewrite this query to improve retrieval:
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )

    new_query = response.choices[0].message.content.strip()

    print(f"New Query: {new_query}")

    return {"query": new_query}


# ✨ Generate Final Answer
def generate_node(state: GraphState):
    print("\n--- GENERATING FINAL ANSWER ---")

    prompt = f"""
    Use the context below to answer the query.

    Context:
    {state['context']}

    Query:
    {state['query']}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response.choices[0].message.content

    return {"prediction": answer}


# 3. DECISION LOGIC
def decide_to_generate(state: GraphState):
    print("\n--- DECISION ---")

    if state["relevance_score"] == "yes":
        print("Context is relevant → Generating answer")
        return "generate"

    elif state["iterations"] >= 3:
        print("Max iterations reached → Generating anyway")
        return "generate"

    else:
        print("Context not relevant → Rewriting query")
        return "rewrite"


# 4. BUILD GRAPH
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("rewrite", rewrite_node)
workflow.add_node("generate", generate_node)

# Set entry point
workflow.set_entry_point("retrieve")

# Edges
workflow.add_edge("retrieve", "grade")

workflow.add_conditional_edges(
    "grade",
    decide_to_generate,
    {
        "generate": "generate",
        "rewrite": "rewrite",
    },
)

workflow.add_edge("rewrite", "retrieve")
workflow.add_edge("generate", END)

# Compile app
app = workflow.compile()

In [ ]:
# 1. Define your test query
test_query = "Suggest a movie where a character struggles with their own modesty or overconfidence."

# 2. Initialize the state
initial_state = {
    "query": test_query,
    "iterations": 0
}

# 3. Run the Graph
print("🚀 Starting LangGraph Test...")
print(f"User Query: {test_query}\n")

for output in app.stream(initial_state):
    
    for node_name, state_update in output.items():
        print(f"📍 Finished Node: {node_name}")
        
        if "prediction" in state_update:
            final_answer = state_update["prediction"]

print("\n" + "="*50)
print("🎬 FINAL AGENTIC RESPONSE:")
print("="*50)
print(final_answer)

# Adding more functionality 
* added decompose node for sub-query implimentation for complex reasoning.
* relaxing prompt for grade_node such that it is not stick to  exact metadata and consider approximation also
* implimenting seaarch through Gradio UI

In [ ]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from groq import Groq
import json

client = Groq(api_key="api_key_here")  

# 1. UPDATED STATE
class GraphState(TypedDict):
    query: str           
    sub_queries: List[str] 
    current_step: int    
    cumulative_ans: str  
    context: str         
    prediction: str
    relevance_score: str
    iterations: int

# 2. NEW & UPDATED NODES

# 🧩 Decompose Node: Breaks complex query into sub-queries
def decompose_node(state: GraphState):
    print("\n--- ANALYZING QUERY & EXTRACTING FILTERS ---")
    prompt = f"""
    Analyze: "{state['query']}"
    Extract JSON:
    - year_start (int)
    - year_end (int)
    - genre (Noir, Comedy, Sci-Fi, Horror, Crime, Action, Drama, Thriller)
    JSON ONLY. Use 0 and "None" if not found.
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        response_format={ "type": "json_object" }
    )
    meta = json.loads(response.choices[0].message.content)

    filters = {}
    if meta['year_start'] != 0:
        filters["year"] = {"$gte": meta['year_start'], "$lte": meta['year_end']}
    if meta['genre'] != "None":
        filters["genre"] = meta['genre']

    return {
        "sub_queries": ["Analyze plot themes", "Verify character consequences"],
        "filters": filters,
        "current_step": 0,
        "iterations": 0
    }
# 🔍 Multi-Step Retrieval: Processes sub-queries one by one
def sub_query_node(state: GraphState):
    step = state["current_step"]
    current_q = state["sub_queries"][step]
    print(f"\n--- PROCESSING SUB-QUERY {step+1}: {current_q} ---")

    # Search vector DB with the sub-query + what we've learned so far
    search_term = f"{current_q} Context: {state['cumulative_ans']}"
    docs = vectorstore.similarity_search(search_term, k=2)
    new_info = "\n".join([d.page_content for d in docs])

    # Combine new info into the cumulative knowledge
    return {
        "cumulative_ans": state["cumulative_ans"] + f"\nStep {step+1} Info: " + new_info,
        "current_step": step + 1
    }

# 🎯 Final Retrieval: Uses the combined knowledge to find the most relevant chunk
def final_retrieve_node(state: GraphState):
    print("\n--- SELECTING FINAL RELEVANT CHUNKS (FUZZY) ---")

    # We use the filters generated in the decompose node
    # If a year was found, we "relax" it by +/- 5 years automatically
    current_filters = state.get("filters", {})

    if "year" in current_filters:
        orig_start = current_filters["year"]["$gte"]
        orig_end = current_filters["year"]["$lte"]
        # Relaxing the boundaries
        current_filters["year"] = {"$gte": orig_start - 5, "$lte": orig_end + 5}
        print(f"Applied Fuzzy Filter: {orig_start-5} to {orig_end+5}")

    # Search using the accumulated knowledge and fuzzy filters
    docs = vectorstore.similarity_search(
        state["cumulative_ans"],
        k=3,
        filter=current_filters if current_filters else None
    )

    context = "\n".join([d.page_content for d in docs])
    return {"context": context}

def grade_node(state: GraphState):
    print("\n--- GRADING RELEVANCE (RELAXED MODE) ---")

    prompt = f"""
    You are a cinematic logic expert.
    Target Query: {state['query']}
    Retrieved Plot: {state['context']}

    CRITERIA:
    1. Does the plot match the THEME (e.g., morality leading to tragedy)?
    2. Is the time period or genre reasonably close (e.g., late 70s vs early 80s)?

    If the plot is a strong thematic match but has minor metadata discrepancies (like the year being 1980 instead of 1979),
    answer YES so the generator can explain the nuance.

    Answer ONLY: YES or NO
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )
    score = response.choices[0].message.content.strip().lower()
    # Normalize score to just 'yes' or 'no'
    relevance = "yes" if "yes" in score else "no"

    return {"relevance_score": relevance}

# 3. UPDATED DECISION LOGIC
def should_continue_steps(state: GraphState):
    if state["current_step"] < len(state["sub_queries"]):
        return "next_step"
    return "final_retrieve"

def generate_node(state: GraphState):
    print("\n--- GENERATING FINAL ANSWER ---")

    prompt = f"""
    Using the provided context, answer the user's movie query.
    If the movie found is slightly outside the requested year (e.g., 1980 instead of 1979),
    explain that it is a close thematic match.

    Query: {state['query']}
    Context: {state['context']}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )

    return {"prediction": response.choices[0].message.content}

# 1. Update the Decision Logic to point to a REWRITE node
def decide_after_grading(state: GraphState):
    print("\n--- AGENTIC EVALUATION ---")
    if state["relevance_score"] == "yes":
        return "generate"
    if state["iterations"] >= 3:
        return "generate"

    # If we fail, don't just go to 'process_step', go to 'rewrite'
    return "research_more"

# 2. Add a specialized Rewrite Node for the Multi-Step flow
def agentic_rewrite_node(state: GraphState):
    print("\n--- AGENT IS RE-PLANNING SEARCH ---")
    prompt = f"""
    The previous research failed to find a perfect match.
    Original Query: {state['query']}
    Current Knowledge: {state['cumulative_ans']}

    Based on what we missed, write ONE highly specific search query
    to find the missing movie in the vector database.
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )
    new_q = response.choices[0].message.content.strip()

    # We replace the sub-query list with this new targeted query
    return {
        "sub_queries": [new_q],
        "current_step": 0, # Reset step so it can process this new query
        "iterations": state["iterations"] + 1
    }



# 4. BUILD THE ENHANCED GRAPH
workflow = StateGraph(GraphState)

workflow.add_node("agentic_rewrite", agentic_rewrite_node)
workflow.add_node("decompose", decompose_node)
workflow.add_node("process_step", sub_query_node)
workflow.add_node("final_retrieve", final_retrieve_node)
workflow.add_node("grade", grade_node) 
workflow.add_node("generate", generate_node) 

workflow.set_entry_point("decompose")

# Standard Flow
workflow.add_edge("decompose", "process_step")
workflow.add_edge("process_step", "final_retrieve")
workflow.add_edge("final_retrieve", "grade")

workflow.add_conditional_edges(
    "grade",
    decide_after_grading,
    {
        "generate": "generate",
        "research_more": "agentic_rewrite" # Go to rewrite instead of process_step
    }
)

workflow.add_edge("agentic_rewrite", "process_step")
workflow.add_edge("generate", END)
app = workflow.compile()


In [ ]:
import gradio as gr

def run_agent(user_query):
    # 1. Initialize State
    initial_state = {
        "query": user_query,
        "current_step": 0,
        "iterations": 0,
        "cumulative_ans": ""
    }

    partial_logs = ""
    final_prediction = "No answer generated."

    # 2. Stream the Graph
    # We yield updates so the UI feels responsive
    for output in app.stream(initial_state):
        for node_name, state_update in output.items():
            # Create a pretty log of what the agent is doing
            status_msg = f"\n📍 [Node: {node_name.upper()}]\n"

            if "sub_queries" in state_update:
                status_msg += f"📋 Plan: {state_update['sub_queries']}\n"
            if "relevance_score" in state_update:
                status_msg += f"⚖️ Relevance: {state_update['relevance_score'].upper()}\n"

            partial_logs += status_msg

            # Capture the final answer when it arrives
            if "prediction" in state_update:
                final_prediction = state_update["prediction"]

            # Yield the logs to the 'Process' box and the answer to the 'Answer' box
            yield partial_logs, final_prediction

# 3. BUILD THE UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 Agentic Movie Researcher (NTCIR-19)")
    gr.Markdown("Identify complex movie plots using Multi-Step LangGraph Reasoning.")

    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Textbox(
                label="Enter your complex movie query",
                placeholder="e.g., A 1970s crime film where a good deed goes wrong...",
                lines=3
            )
            submit_btn = gr.Button("🔍 Start Research", variant="primary")

        with gr.Column(scale=2):
            answer_output = gr.Markdown(label="Final Agent Report")

    with gr.Accordion("🕵️ Agent Thought Process (Logs)", open=True):
        log_output = gr.Code(label="Internal Reasoning Steps", language="markdown")

    # Link the button to the function
    submit_btn.click(
        fn=run_agent,
        inputs=query_input,
        outputs=[log_output, answer_output]
    )

# 4. LAUNCH
demo.queue().launch(debug=True)

/tmp/ipykernel_5129/3381116183.py:37: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://adcaa040899789c3ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



--- ANALYZING QUERY FOR METADATA FILTERS ---

--- PROCESSING SUB-QUERY 1: Research plot details... ---

--- SELECTING FINAL RELEVANT CHUNKS ---

--- GRADING RELEVANCE ---

--- AGENTIC EVALUATION ---

--- AGENT IS RE-PLANNING SEARCH ---

--- PROCESSING SUB-QUERY 1: To find the 1970s crime or neo-noir movie that matches the given description, the following highly specific search query can be used:

"(1970s film) AND (protagonist attempts moral act) AND (act leads to violent consequences) AND (involves piracy OR kidnapping OR brainwashing) AND (father-son relationship) AND (island setting OR Bermuda Triangle)"


Alternatively, a more concise and precise query could be:

"1970s neo-noir film where protagonist's moral act sparks violent chain of events, featuring father-son duo, piracy, and island captivity" ---

--- SELECTING FINAL RELEVANT CHUNKS ---

--- GRADING RELEVANCE ---

--- AGENTIC EVALUATION ---

--- AGENT IS RE-PLANNING SEARCH ---

--- PROCESSING SUB-QUERY 1: To find the movie 

# Self check before Gradio Implimentation

In [ ]:
# 1. Define a complex, multi-layered query
test_query = """
Find a 1970s crime or neo-noir movie where the protagonist tries to do
something morally right, but that specific act of 'goodness' triggers
a chain of violent or tragic events for them.
"""

# 2. Initialize the state
initial_state = {
    "query": test_query,
    "current_step": 0,
    "iterations": 0,
    "cumulative_ans": ""
}

# 3. Run the Agentic Graph
print("🚀 Launching Multi-Step Agentic Research...")
print(f"Main Task: {test_query}\n")


for output in app.stream(initial_state):
    for node_name, state_update in output.items():
        print(f"📍 Node Completed: {node_name}")

        
        if "sub_queries" in state_update:
            print(f"📋 Research Plan: {state_update['sub_queries']}")

        
        if "cumulative_ans" in state_update:
            print("📝 Updated Knowledge Base (Internal Context Created)")

        if "prediction" in state_update:
            final_ans = state_update["prediction"]

print("\n" + "="*60)
print("🎬 FINAL RESEARCH REPORT:")
print("="*60)
print(final_ans)